# CodeAlpha - Disease Prediction Model
Predicts whether a tumor is **Malignant** or **Benign** (breast cancer diagnosis) using Logistic Regression, Random Forest, and SVM, based on patient diagnostic measurements.

**How to use this notebook:**
1. Run each cell in order (click the cell, press `Shift + Enter`)
2. Charts and metrics will display right below each cell
3. Download generated files at the end using the last cell


## Step 1: Install libraries
(Colab already has these pre-installed — this just makes sure versions are compatible.)

In [1]:
!pip install -q pandas numpy scikit-learn matplotlib seaborn joblib

## Step 2: Load the dataset
We use the **Wisconsin Breast Cancer Diagnostic dataset**, built directly into scikit-learn (no download needed). It contains 569 patient records with 30 numeric features computed from digitized images of breast tissue (radius, texture, smoothness, concavity, etc.), along with the diagnosis label.

In [2]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
# target: 0 = malignant, 1 = benign (as per sklearn's encoding)
df["diagnosis"] = data.target

df.to_csv("disease_data.csv", index=False)
print(f"Loaded dataset with {len(df)} rows and {df.shape[1]-1} features. Saved to disease_data.csv")
print("\nClass balance (0 = Malignant, 1 = Benign):")
print(df["diagnosis"].value_counts(normalize=True).round(3))
df.head()


Loaded dataset with 569 rows and 30 features. Saved to disease_data.csv

Class balance (0 = Malignant, 1 = Benign):
diagnosis
1    0.627
0    0.373
Name: proportion, dtype: float64


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,diagnosis
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


## Step 3: Quick exploratory look
A correlation heatmap of a few key features against the diagnosis, just to sanity-check the data before modeling.

In [3]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

key_features = ["mean radius", "mean texture", "mean smoothness", "mean concavity",
                 "mean compactness", "mean symmetry", "diagnosis"]
corr = df[key_features].corr()

plt.figure(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation of Key Features with Diagnosis")
plt.tight_layout()
plt.savefig("correlation_heatmap.png", dpi=120)
plt.show()


## Step 4: Train and evaluate 3 models
Logistic Regression, Random Forest, and SVM — compared using Accuracy, Precision, Recall, F1-Score, and ROC-AUC.

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)
import joblib

RANDOM_STATE = 42

def evaluate_model(name, model, X_test, y_test, results):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    metrics = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_proba),
    }
    results.append(metrics)
    divider = "=" * 55
    print(f"\n{divider}\n{name}\n{divider}")
    print(classification_report(y_test, y_pred, target_names=["Malignant", "Benign"]))
    print(f"ROC-AUC: {metrics['ROC-AUC']:.4f}")
    return y_pred, y_proba, metrics

target = "diagnosis"
X = df.drop(columns=[target])
y = df[target]
feature_names = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

results = []
roc_data = {}
cms, names = [], []

log_reg = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
log_reg.fit(X_train_scaled, y_train)
y_pred, y_proba, _ = evaluate_model("Logistic Regression", log_reg, X_test_scaled, y_test, results)
fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_data["Logistic Regression"] = (fpr, tpr, roc_auc_score(y_test, y_proba))
cms.append(confusion_matrix(y_test, y_pred)); names.append("Logistic Regression")

rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=RANDOM_STATE)
rf.fit(X_train, y_train)
y_pred, y_proba, _ = evaluate_model("Random Forest", rf, X_test, y_test, results)
fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_data["Random Forest"] = (fpr, tpr, roc_auc_score(y_test, y_proba))
cms.append(confusion_matrix(y_test, y_pred)); names.append("Random Forest")

svm = SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE)
svm.fit(X_train_scaled, y_train)
y_pred, y_proba, _ = evaluate_model("SVM", svm, X_test_scaled, y_test, results)
fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_data["SVM"] = (fpr, tpr, roc_auc_score(y_test, y_proba))
cms.append(confusion_matrix(y_test, y_pred)); names.append("SVM")

results_df = pd.DataFrame(results).sort_values("ROC-AUC", ascending=False)
divider = "=" * 55
print(f"\n{divider}\nMODEL COMPARISON (sorted by ROC-AUC)\n{divider}")
print(results_df.to_string(index=False))
results_df.to_csv("model_comparison_results.csv", index=False)
results_df



Logistic Regression
              precision    recall  f1-score   support

   Malignant       0.98      0.98      0.98        42
      Benign       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114

ROC-AUC: 0.9954

Random Forest
              precision    recall  f1-score   support

   Malignant       0.95      0.93      0.94        42
      Benign       0.96      0.97      0.97        72

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114

ROC-AUC: 0.9940

SVM
              precision    recall  f1-score   support

   Malignant       0.98      0.98      0.98        42
      Benign       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
w

,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Logistic Regression,0.982456,0.986111,0.986111,0.986111,0.995370
2,SVM,0.982456,0.986111,0.986111,0.986111,0.995040
1,Random Forest,0.956140,0.958904,0.972222,0.965517,0.994048


## Step 5: View confusion matrices

In [5]:
fig, axes = plt.subplots(1, len(cms), figsize=(5 * len(cms), 4))
for ax, cm, name in zip(axes, cms, names):
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["Malignant", "Benign"], yticklabels=["Malignant", "Benign"])
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=120)
plt.show()


## Step 6: View ROC curve comparison

In [6]:
plt.figure(figsize=(7, 6))
for name, (fpr, tpr, auc) in roc_data.items():
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Random guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves - Model Comparison")
plt.legend()
plt.tight_layout()
plt.savefig("roc_curves.png", dpi=120)
plt.show()


## Step 7: View feature importance (Random Forest)

In [7]:
importances = rf.feature_importances_
idx = np.argsort(importances)[::-1][:15]
plt.figure(figsize=(8, 6))
sns.barplot(x=importances[idx], y=np.array(feature_names)[idx], color="seagreen")
plt.title("Top 15 Feature Importances - Random Forest")
plt.xlabel("Importance")
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=120)
plt.show()


## Step 8: Save the best model

In [8]:
best_model_name = results_df.iloc[0]["Model"]
best_model = {"Logistic Regression": log_reg, "Random Forest": rf, "SVM": svm}[best_model_name]
joblib.dump(best_model, "best_disease_model.pkl")
joblib.dump(scaler, "scaler.pkl")
print(f"Best model: {best_model_name} - saved to best_disease_model.pkl")


Best model: Logistic Regression - saved to best_disease_model.pkl


## Step 9: Download your files
Run this cell to download everything to your computer (or grab them from the Files panel on the left).

In [9]:
from google.colab import files

for fname in ["disease_data.csv", "model_comparison_results.csv", "correlation_heatmap.png",
              "confusion_matrices.png", "roc_curves.png", "feature_importance.png",
              "best_disease_model.pkl", "scaler.pkl"]:
    files.download(fname)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>